In [ ]:
# ==============================================================================
# CELULA 1: CONFIGURACAO DE AMBIENTE E HARDWARE (LOCAL)
# ==============================================================================
# DIAGRAMA DE ARQUITETURA DE HARDWARE E DEPENDÊNCIAS
# ------------------------------------------------------------------------------
#  [ Sistema Operacional Host ] -> Isola o Kernel do Sistema
#          |
#          v
#  [ Ambiente Virtual (Venv/Conda) ] -> Garante Paridade Local vs. Nuvem (HPC)
#          |
#          +---> [ NumPy ] ---------> Manipulação de Matrizes CPU (Row-Major)
#          +---> [ PyTorch ] -------> Computação Tensorial e Autograd (Gradientes)
#          +---> [ Deepwave ] ------> Resolvedor Numérico FDTD da Equação da Onda
#          |
#          v
#  [ Alocador de Hardware: torch.device ]
#          |
#          +---> Se CUDA disponível ----> Aloca Tensores na VRAM da GPU (Aceleração)
#          +---> Se CUDA ausente -------> Aloca Tensores na RAM da CPU (Fallback)
# ==============================================================================

import os
import sys
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from typing import Tuple, Any

import deepwave
from deepwave import scalar

# Verificacao de Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Sistema] Executando localmente. Dispositivo mapeado: {device}")

In [ ]:
# ==============================================================================
# CELULA 2: CONSTRUCAO E VISUALIZACAO DO MODELO EM NUMPY
# ==============================================================================
# DIAGRAMA GEOMÉTRICO DO DOMÍNIO SINTÉTICO DISCRETO
# ------------------------------------------------------------------------------
#   Origem: (oy=0.0, oz=0.0)
#          +-------------------------------------------------------+
#          | Camada Superior ( layer_01 = 1500.0 m/s )              |
#          | Ex: Sedimento Raso / Água                             |
#          |                                                       |  Profundidade (Z)
#          |                                                       |  Discretizada:
#  z=350m  |=======================================================|  nz = 48 nós
#  (nó 35) | Interface do Refletor Geológico (Descontinuidade)     |  dz = 10.0 metros
#          |-------------------------------------------------------|
#          | Embasamento Profundo ( layer_02 = 2000.0 m/s )        |
#          | Ex: Alvo da Reflexão Sísmica                          |
#          +-------------------------------------------------------+
#                                    Largura (Y)
#                    Discretizada: ny = 96 nós | dy = 10.0 metros
#                     Extensão Espacial Física: 0 a 950 metros
# ==============================================================================

# Definicoes das dimensoes do modelo
ny = 96
nz = 48
oz = 0.0
oy = 0.0
dz = 10.0
dy = 10.0

# Construcao dos vetores de profundidade (z) e distancia (y)
z_values = np.arange(oz, oz + nz * dz, dz)
y_values = np.arange(oy, oy + ny * dy, dy)

layer_01 = 1500.0
layer_02 = 2000.0

# Criacao do modelo homogeneo em NumPy
model = (np.ones((nz, ny)) * layer_01).astype(np.float32)

# Adicionando a segunda camada (refletor)
model[35:, :] = layer_02

# Obtencao dos valores maximos e minimos para escala de cor
maxval = np.max(model)
minval = np.min(model)

# Plotagem do modelo original (NumPy)
plt.figure(figsize=(6, 4))
plt.imshow(model, cmap='jet', extent=[y_values[0], y_values[-1], z_values[-1], z_values[0]])
plt.clim(vmin=minval, vmax=maxval)

cbar = plt.colorbar(shrink=0.5)
cbar.ax.tick_params(labelsize=10)
tick_values = np.arange(minval, maxval + 1, 250)
cbar.set_ticks(tick_values)

plt.title('True model (NumPy)')
plt.xlabel('Distance (m)')
plt.ylabel('Depth (m)')
cbar.set_label(r'$V_p$ (m/s)', labelpad=-18, y=1.16, rotation=0)
plt.show()

### Justificativa Arquitetural: O Modelo Inicial (Flat / Homogeneo)

A instanciacao do tensor `model_homo` com uma velocidade constante (1400 m/s) e desprovido de refletores atua como o modelo de *background* (chute inicial cego) para o problema inverso. 

Na topologia de Full Waveform Inversion (FWI):
1. O algoritmo assume desconhecimento total da subsuperficie real.
2. A propagacao direta (modelagem via FDTD) e executada neste modelo homogeneo.
3. O motor de derivacao automatica (Autograd) calcula o residuo (a diferenca) entre o dado modelado e o dado observado em superficie.
4. O residuo e injetado nos receptores e retropropagado (Estado Adjunto) para extrair o Gradiente FWI.

O gradiente aponta as coordenadas espaciais exatas onde o modelo homogeneo inicial devera ser atualizado pela Rede Neural (PINN) nas futuras iteracoes do otimizador.

In [ ]:
# ==============================================================================
# CELULA 3: CONVERSAO DO MODELO NUMPY PARA TENSORES PYTORCH
# ==============================================================================
# DIAGRAMA DE FLUXO E ROTAÇÃO TENSORIAL: CPU vs. GPU
# ------------------------------------------------------------------------------
#  [ Matriz NumPy (RAM) ] --------> Formato: Row-Major (nz, ny) -> (48, 96)
#            |
#            v  ( torch.tensor() & .device )
#  [ Tensor PyTorch (VRAM) ] -----> Alocado na GPU -> Mantém (48, 96)
#            |
#            v  ( Operador de Transposição: .T )
#  [ Tensor Rotacionado ] --------> Formato: Column-Major (ny, nz) -> (96, 48)
#            |                      *Exigência estrita do motor Deepwave*
#            |
#            +---> [ model_true ] -> Meio real com a descontinuidade (2000 m/s)
#            |
#            +---> [ model_homo ] -> Chute Inicial Cego / Background (1400 m/s)
#                                    *Gerador do Resíduo FWI Primário*
# ==============================================================================

print(f"NumPy model shape : {model.shape}") # Formato original Row-Major (Z, Y)

# ------------------------------------------------------------------------------
# Compatibilizacao Tensorial (NumPy -> PyTorch/Deepwave)
# O motor Deepwave exige a orientacao de eixos espaciais no formato (Y, Z).
# O operador .T rotaciona a matriz na memoria da GPU (de 48x96 para 96x48),
# garantindo que a propagacao FDTD interprete profundidade e largura corretamente.
# ------------------------------------------------------------------------------
model_true = torch.tensor(model, dtype=torch.float32, device=device).T

# ------------------------------------------------------------------------------
# Construcao do Modelo Inicial (Background / Chute Inicial)
# A inversao FWI exige um ponto de partida cego. Instanciamos um meio 
# 100% homogeneo (sem refletores) com velocidade constante de 1400 m/s. 
# Esse distanciamento proposital da realidade e o que gerara o residuo matematico.
# ------------------------------------------------------------------------------
model_homo = (torch.ones(ny, nz, device=device))
model_homo = model_homo * (layer_01 - 100) 

print(f"Torch model shape : {model_true.shape}") # Formato corrigido para propagacao (96, 48)

# ------------------------------------------------------------------------------
# Inspecao Visual (Plotagem Comparativa)
# O metodo .cpu() e obrigatorio, pois o matplotlib nao tem acesso a memoria 
# VRAM da placa de video, exigindo que o tensor seja copiado para a memoria RAM.
# ------------------------------------------------------------------------------
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(model_true.cpu(), aspect='auto', cmap='jet')
plt.title(f"Torch model_true\nshape={tuple(model_true.shape)}")

plt.subplot(1, 2, 2)
plt.imshow(model_homo.cpu(), aspect='auto', cmap='jet')
plt.title(f"Torch model_homo\nshape={tuple(model_homo.shape)}")

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELULA 4: PARAMETROS DO DEEPWAVE E GEOMETRIA DE AQUISICAO
# ==============================================================================
# DIAGRAMA LINEAR DO ARRANJO DE FONTES E RECEPTORES SÍSMICOS
# ------------------------------------------------------------------------------
#   Superfície do Grid (Profundidade Z = 10 nós / 100 metros)
#   Z=0m  -------------------------------------------------------------
#         |                                                           |
#   Z=100m|-------- [Fonte (X=20)] .............. [Receptor (X=76)] --|
#         |           (Tiro Ativo)                   (Geofone)        |
#         |                                                           |
#         |                      Subsuperfície                        |
# ==============================================================================

# --- Parametros da Malha (Grid) e Discretizacao Temporal ---
dx = 10.0               # Espacamento espacial do grid em metros (assume dz = dy)
dt = 0.004              # Passo de tempo da simulacao em segundos (condicao CFL)
nt = 480                # Numero total de iteracoes no tempo (janela de registro de 1.92s)

# --- Parametros da Fonte (Assinatura Sismica) ---
peak_freq = 15          # Frequencia central da wavelet de Ricker em Hz
peak_source_time = 1 / peak_freq
peak_time = 1.0 / peak_freq  # Atraso temporal para centralizar o pico maximo de energia da wavelet

# --- Topologia da Aquisicao ---
num_dims = 2                 # Dimensionalidade do problema (2D)
num_shots = 1                # Quantidade total de tiros (explosoes virtuais)
num_sources_per_shot = 1     # Quantidade de fontes ativas simultaneamente por tiro
num_receivers_per_shot = 1   # Quantidade de geofones gravando por tiro
source_spacing = 1           # Espacamento entre tiros adjacentes (em nos do grid)
receiver_spacing = 1         # Espacamento entre receptores adjacentes (em nos do grid)
first_source = 20            # Indice X (lateral) onde o arranjo de fontes comeca
first_receiver = 76          # Indice X (lateral) onde o arranjo de receptores comeca
source_depth = 10            # Profundidade (Z) constante da fonte em nos do grid
receiver_depth = 10          # Profundidade (Z) constante dos geofones em nos do grid

# ------------------------------------------------------------------------------
# Configuracao de Fontes (Mapeamento Tensorial)
# O tensor exige formato: [quantidade_tiros, fontes_por_tiro, 2 coordenadas (X, Z)]
# ------------------------------------------------------------------------------
source_locations = torch.zeros(num_shots, num_sources_per_shot, 2, dtype=torch.long, device=device)
source_locations[..., 1] = source_depth  # Atribui a coordenada Z (profundidade) para todas as fontes
source_locations[:, 0, 0] = torch.arange(num_shots) * source_spacing + first_source  # Atribui a coordenada X

# ------------------------------------------------------------------------------
# Configuracao de Receptores (Mapeamento Tensorial)
# O tensor exige formato: [quantidade_tiros, receptores_por_tiro, 2 coordenadas (X, Z)]
# ------------------------------------------------------------------------------
receiver_locations = torch.zeros(num_shots, num_receivers_per_shot, 2, dtype=torch.long, device=device)
receiver_locations[..., 1] = receiver_depth  # Atribui a coordenada Z para todos os geofones
receiver_locations[:, :, 0] = ((torch.arange(num_receivers_per_shot) * receiver_spacing + first_receiver).repeat(num_shots, 1))

# ------------------------------------------------------------------------------
# Geracao do pulso da fonte (Wavelet de Ricker)
# Formato exigido: [quantidade_tiros, fontes_por_tiro, passos_de_tempo]
# ------------------------------------------------------------------------------
source_amplitudes = (
    deepwave.wavelets.ricker(peak_freq, nt, dt, peak_time)
    .repeat(num_shots, num_sources_per_shot, 1)
    .to(device)
)

# Plotagem do pulso da fonte (Corrigido para reducao dimensional via squeeze)
plt.figure(figsize=(6, 3))
plt.plot(source_amplitudes.cpu().squeeze().numpy())
plt.ylim(-1, 1 + 0.1)
plt.xlim(0, nt)
plt.title('Source pulse (Ricker wavelet)')
plt.ylabel('Amplitude')
plt.xlabel('Time sample')
plt.show()

In [ ]:
# ==============================================================================
# CELULA 5: MODELAGEM DIRETA E CAPTURA DE SNAPSHOTS ASSÍNCRONOS
# ==============================================================================
# DIAGRAMA DE FLUXO DA MODELAGEM DIRETA (ANTI-CAIXA PRETA)
# ------------------------------------------------------------------------------
#        [Wavelet de Ricker] -> Injeção em Coordenadas X_s (Z=100m)
#                                       |
#                                       v
#        ================= [FDTD Engine (Deepwave)] =================
#        |                                                          |
#        |   Camada 1 (Vp = 1500m/s) -> Propagação Linear           |
#        |   ------------------- Interface de Reflexão ------------ |
#        |   Camada 2 (Vp = 2000m/s) -> Geração de Eco/Reflexão     |
#        |                                                          |
#        ============================================================
#                                       |
#                                       v
#      [Sismograma] <- Registro Contínuo em Coordenadas X_r (Z=100m)
# ==============================================================================

print("[Modelagem] Propagando no modelo verdadeiro...")

# A funcao 'scalar' encapsula o solver FDTD (Diferencas Finitas no Dominio do Tempo). 
# Executa a simulacao de ponta a ponta na GPU abstraindo o loop temporal (passos nt).
out_true = scalar(
    model_true,                            # Tensor com o campo de velocidades (meio geologico)
    dx, dt,                                # Malha de discretizacao no espaco e no tempo
    max_vel=2500.0,                        # Velocidade limite para condicao de estabilidade de Courant (CFL)
    source_amplitudes=source_amplitudes,   # Energia acustica injetada no sistema (Wavelet de Ricker)
    source_locations=source_locations,     # Coordenadas ativas (posicao de disparo)
    receiver_locations=receiver_locations, # Geometria de gravacao (posicao dos geofones virtuais)
    accuracy=8,                            # Ordem do estencil espacial (computa 8 pontos vizinhos simultaneamente)
    pml_freq=peak_freq,                    # Frequencia central para calibrar o amortecimento na borda
    pml_width=[30, 30, 30, 30]             # Perfectly Matched Layers: bordas de 30 pontos para anular eco numerico
)

# O motor FDTD retorna uma tupla com multiplos estados da propagacao. 
# O indice [-1] extrai estritamente a matriz do sismograma gravado nos receptores.
receiver_amplitudes_true = out_true[-1]


print("[Modelagem] Propagando no modelo homogeneo (background)...")

# A mesma instancia do motor FDTD, agora injetando a onda no modelo "cego" (sem refletores).
out_homo = scalar(
    model_homo, 
    dx, dt, max_vel=2500.0,
    source_amplitudes=source_amplitudes,
    source_locations=source_locations,
    receiver_locations=receiver_locations,
    accuracy=8,
    pml_freq=peak_freq, pml_width=[30, 30, 30, 30]
)

# Extracao do sismograma sintetico que sera comparado ao sismograma verdadeiro para gerar o residuo.
receiver_amplitudes_homo = out_homo[-1]

# Calculo do residuo dos dados
residual = receiver_amplitudes_true - receiver_amplitudes_homo
print(f"Formato do Residuo: {residual.shape}")

# Extracao de um traco para o tiro central
ishot = num_shots // 2
print(f"Central shot index: {ishot}")

time_vec = np.arange(nt) * dt
trace_true = receiver_amplitudes_true[ishot].detach().cpu().mT
trace_homo = receiver_amplitudes_homo[ishot].detach().cpu().mT
trace_res  = residual[ishot].detach().cpu().mT

# Escala de cores comum baseada no maximo absoluto
vmax_trace = max(
    float(torch.max(torch.abs(trace_true))),
    float(torch.max(torch.abs(trace_homo))),
    float(torch.max(torch.abs(trace_res)))
)
vmin_trace = -vmax_trace

# Plotagem dos Sismogramas (True, Initial, Residual)
fig, ax = plt.subplots(1, 3, figsize=(12, 5), sharey=True)

ax[0].imshow(trace_true.numpy(), aspect='auto', cmap='gray', vmin=vmin_trace, vmax=vmax_trace, extent=[0, 1, time_vec[-1], time_vec[0]])
ax[0].set_title(f"True model\n(shot {ishot})")
ax[0].set_ylabel("Time (s)")
ax[0].set_xticks([])

ax[1].imshow(trace_homo.numpy(), aspect='auto', cmap='gray', vmin=vmin_trace, vmax=vmax_trace, extent=[0, 1, time_vec[-1], time_vec[0]])
ax[1].set_title(f"Initial model\n(shot {ishot})")
ax[1].set_xticks([])

ax[2].imshow(trace_res.numpy(), aspect='auto', cmap='gray', vmin=vmin_trace, vmax=vmax_trace, extent=[0, 1, time_vec[-1], time_vec[0]])
ax[2].set_title(f"Residual\n(shot {ishot})")
ax[2].set_xticks([])

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELULA 6: CALCULO DE GRADIENTE ISOLADO (AUTOGRAD)
# ==============================================================================
# DIAGRAMA DE FLUXO: MÉTODO DO ESTADO ADJUNTO (AUTOGRAD FWI)
# ------------------------------------------------------------------------------
#  [ Modelo Inicial (Background) ] ---> (requires_grad_=True)
#            |
#            v
#  [ Propagação Direta (FDTD) ] ------> Gera: Sismograma Sintético
#                                                  |
#  [ Sismograma Real (Observado) ]                 v
#            |                              [ Cálculo do Resíduo (MSE) ]
#            +------------------------------------->|
#                                                   v
#  [ Gradiente FWI (Atualização) ] <------- [ loss.backward() ]
#  (Retropropagação no Tempo Reverso cruzada com o Campo Direto)
# ==============================================================================

def gradient_DW(model_input: torch.Tensor) -> Tuple[np.ndarray, float]:
    """
    Motor de Extracao do Gradiente FWI.
    Calcula a derivada da funcao objetivo em relacao ao modelo de velocidade
    utilizando a engine Autograd (que emula o Metodo do Estado Adjunto).
    """
    
    # --------------------------------------------------------------------------
    # Rastreamento Tensorial (Grafo Computacional)
    # Clonamos o modelo de entrada para isolar a operacao na memoria.
    # O comando requires_grad_(True) e o gatilho que avisa a GPU para rastrear
    # cada operacao matematica e construir o grafo de derivadas parciais.
    # --------------------------------------------------------------------------
    vel = model_input.clone().detach().to(device).requires_grad_(True)
    
    # Metrica de Erro: Mean Squared Error (MSE) no dominio dos dados (Sismogramas)
    objfun = torch.nn.MSELoss()

    # Propagacao Direta (Forward Pass) no modelo atual
    out = scalar(
        vel, dx, dt, max_vel=2500.0,
        source_amplitudes=source_amplitudes,
        source_locations=source_locations,
        receiver_locations=receiver_locations,
        accuracy=8,
        pml_freq=peak_freq, pml_width=[30, 30, 30, 30]
    )

    # --------------------------------------------------------------------------
    # Retropropagacao e Estado Adjunto (Backward Pass)
    # 1. Calculamos o residuo (a diferenca entre o sismograma modelado e o real).
    # 2. loss.backward() injeta esse residuo nos receptores virtuais e o 
    #    propaga no tempo reverso atraves do grid, cruzando-o com o 
    #    campo de onda direto para iluminar o Gradiente de Velocidade.
    # --------------------------------------------------------------------------
    loss = objfun(out[-1], receiver_amplitudes_true)
    loss.backward()
    
    # --------------------------------------------------------------------------
    # Condicionamento do Gradiente (Gradient Clipping)
    # A inversao FWI sofre com "source footprints" (singularidades de amplitude 
    # excessiva nas coordenadas adjacentes as fontes). Calculamos o quantil de 
    # 98% e clipamos os valores extremos para evitar a quebra do otimizador.
    # --------------------------------------------------------------------------
    clip_val = torch.quantile(vel.grad.detach().abs(), 0.98)
    torch.nn.utils.clip_grad_value_(vel, clip_val)

    # Extraimos o gradiente da GPU (.cpu), removemos do grafo (.detach) e 
    # transpomos (.T) de volta para o formato de visualizacao do NumPy (Z, Y).
    return vel.grad.detach().cpu().numpy().T, loss.item()

# ------------------------------------------------------------------------------
# Execucao da Extracao e Inspecao Visual do Gradiente Inicial
# ------------------------------------------------------------------------------
grad, loss_val = gradient_DW(model_homo)
print(f"Loss value: {loss_val}")

maxval_g = np.max(grad)
minval_g = np.min(grad)
print(f"Min Grad: {minval_g}, Max Grad: {maxval_g}")

# Renderizacao grafica do mapa topologico do gradiente
plt.figure(figsize=(6, 4))
plt.imshow(grad, cmap='jet', extent=[y_values[0], y_values[-1], z_values[-1], z_values[0]])
plt.clim(vmin=minval_g, vmax=maxval_g)

cbar = plt.colorbar(shrink=0.5)
cbar.ax.tick_params(labelsize=10)
cbar.set_ticks(np.arange(minval_g, maxval_g + maxval_g*0.01, (maxval_g - minval_g)/4))
plt.title('Gradient')
plt.xlabel('Distance (m)')
plt.ylabel('Depth (m)')
plt.show()

In [ ]:
# ==============================================================================
# CELULA 7: PREPARACAO DOS CALLBACKS PARA SNAPSHOTS (AUXILIAR NA GERACAO DO VIDEO EM FORMATO HTML)
# ==============================================================================
# DIAGRAMA DE INTERCEPTAÇÃO ASSÍNCRONA (CALLBACKS)
# ------------------------------------------------------------------------------
#  [ Motor FDTD (Caixa Preta na GPU) ]
#   t=0  ----->  t=1  ----->  t=2  -----> ... -----> t=NT
#    |            |            |
#    v            v            v
#  (Hook)       (Hook)       (Hook)   <-- ForwardCallback / BackwardCallback
#    |            |            |
#    +------------+------------+------> [ Buffer na Memória RAM (Snapshots) ]
#                                       (Preserva a VRAM da GPU contra OOM)
# ==============================================================================

# Frequencia de captura: 1 significa que interceptaremos o estado a cada passo temporal (dt).
callback_frequency = 1

# ------------------------------------------------------------------------------
# Buffers de Armazenamento (Memoria RAM)
# Pre-alocacao de tensores tridimensionais [Tempo, Y, Z] para guardar a evolucao 
# do campo de onda espacial ao longo da simulacao.
# ------------------------------------------------------------------------------
forward_snapshots = torch.zeros(nt // callback_frequency, ny, nz)
backward_snapshots = torch.zeros(nt // callback_frequency, ny, nz)
gradient_snapshots = torch.zeros(nt // callback_frequency, ny, nz)

class ForwardCallback:
    """
    Hook de execucao (Callback) para a propagacao direta.
    
    Nota: O solver FDTD e como uma "caixa preta" que roda na placa 
    de video do inicio ao fim sem parar. O callback funciona como um alarme: 
    ele instrui o motor a pausar a fisica a cada passo de tempo, permitir que 
    este bloco copie a "foto" atual da onda (snapshot) e, entao, retomar a corrida.
    Sem isso, so teriamos acesso ao sismograma final, impossibilitando o video.
    """
    def __init__(self, ishot: int):
        self.ishot = ishot
        self.step = 0

    def __call__(self, state: Any) -> None:
        # .cpu() transfere a matriz da GPU para a RAM do sistema; 
        # .clone() isola o tensor, garantindo que o motor Autograd nao guarde lixo na memoria.
        forward_snapshots[self.step] = state.get_wavefield("wavefield_0")[self.ishot].cpu().clone()
        self.step += 1


class BackwardCallback:
    """
    Hook de execucao (Callback) para a retropropagacao (Estado Adjunto).
    
    Mecanica: Assim como no forward, este callback intercepta o motor rodando 
    de tras para frente. Ele "espiona" o processo para extrair simultaneamente 
    duas coisas: a onda de erro (residuo) voltando no tempo e o mapa do gradiente 
    (a faisca de correlacao) sendo desenhado passo a passo na subsuperficie.
    """
    def __init__(self, ishot: int):
        self.ishot = ishot
        self.step = 0

    def __call__(self, state: Any) -> None:
        # Registra a posicao do residuo fisico viajando de baixo para cima
        backward_snapshots[self.step] = state.get_wavefield("wavefield_0")[self.ishot].cpu().clone()
        
        # Registra a 'faisca' de correlacao (Gradiente) sendo atualizada passo a passo
        gradient_snapshots[self.step] = state.get_gradient("v")[0].cpu().clone()
        self.step += 1

In [ ]:
# ==============================================================================
# CELULA 8: PROPAGACAO DIRETA, EXTRACAO HTML E SNAPSHOT ESTATICO
# ==============================================================================
# DIAGRAMA CINEMÁTICO: PROPAGAÇÃO DIRETA (FORWARD PASS)
# ------------------------------------------------------------------------------
#  Tempo (t) avança: 0s ----------------------------------------> 1.92s
#
#  [ Fonte (Wavelet) ] -> Injeção no Grid (Z=10m)
#          |
#          v
#  [ Frente de Onda ] -> Expansão Esférica no Modelo Verdadeiro
#          |
#          +---> Bate no Refletor (Z=350m) ---> Gera Eco (Reflexão)
#          |
#          v
#  [ Receptores (Z=10m) ] -> Gravam o Sismograma Contínuo
# ==============================================================================

print("[Animacao] Rodando propagacao completa no modelo verdadeiro...")

# Isolamento de memoria e preparacao do tensor para o grafo computacional
vmodel_true = model_true.clone().requires_grad_(True)
vmax_true = model_true.max().item() # Extrai velocidade limite para a condicao CFL

# Seleciona o tiro central para visualizacao (irrelevante aqui pois temos apenas 1 tiro)
ctrl_shot = num_shots // 2

# ------------------------------------------------------------------------------
# Invocacao do Solver com Interceptacao de Memoria (Callback)
# O motor ira injetar a wavelet no grid e, a cada passo de tempo (dt), o 
# ForwardCallback copiara a matriz do campo de onda para o buffer na memoria RAM.
# ------------------------------------------------------------------------------
out_fwd = scalar(
    vmodel_true, dx, dt, max_vel=vmax_true,
    source_amplitudes=source_amplitudes,
    source_locations=source_locations,
    receiver_locations=receiver_locations,
    accuracy=8, pml_freq=peak_freq, pml_width=[30, 30, 30, 30],
    forward_callback=ForwardCallback(ishot=ctrl_shot),
    callback_frequency=callback_frequency
)
data_true = out_fwd[-1]

# ------------------------------------------------------------------------------
# Compilacao da Animacao HTML (Cinematica da Propagacao)
# ------------------------------------------------------------------------------
# Recorte de amplitude (clipping) no quantil 99% para calibracao do Colormap.
# Impede que a energia concentrada na fonte (singularidade) ofusque o resto da onda.
vmax_fwd_anim = torch.quantile(forward_snapshots, 0.99).item()

fig_anim, ax_anim = plt.subplots(figsize=(5, 4))
# A transposicao (.T) retorna o array GPU (Y, Z) para o formato visual de tela (Z, Y)
im_anim = ax_anim.imshow(forward_snapshots[1].T, vmax=vmax_fwd_anim, cmap="seismic", animated=True)
ax_anim.set_xticks([])
ax_anim.set_yticks([])
title_anim = ax_anim.set_title("Forward (t = 0.000 s)")
plt.close(fig_anim) # Suprime a renderizacao estatica prematura da figura base

def update_fwd(frame):
    """Funcao de atualizacao iterativa para o motor de animacao do matplotlib."""
    im_anim.set_data(forward_snapshots[frame].T)
    t = frame * callback_frequency * dt
    title_anim.set_text(f"Forward (t = {t:.3f} s)")
    return (im_anim, title_anim)

# Gera o objeto de animacao e o compila para uma interface HTML5/JavaScript interativa
ani_fwd = animation.FuncAnimation(fig_anim, update_fwd, frames=len(forward_snapshots), interval=25, blit=True)
display(HTML(ani_fwd.to_jshtml()))

# ------------------------------------------------------------------------------
# Visualizacao Estatica de Inspecao
# Renderiza um instante especifico para conferencia no frontend do notebook.
# ------------------------------------------------------------------------------
snap_idx = 10
plt.figure(figsize=(5, 4))
plt.imshow(forward_snapshots[snap_idx].T, cmap="seismic")
plt.colorbar()
plt.title(f"Forward wavefield (snapshot {snap_idx})")
plt.show()

In [ ]:
# ==============================================================================
# CELULA 9: METODO DO ESTADO ADJUNTO (BACKWARD PASS) E ANIMACAO COMPLETA
# ==============================================================================
# DIAGRAMA CINEMÁTICO: PROPAGAÇÃO DIRETA (FORWARD PASS)
# ------------------------------------------------------------------------------
#  Tempo (t) avança: 0s ----------------------------------------> 1.92s
#
#  [ Fonte (Wavelet) ] -> Injeção no Grid (Z=10m)
#          |
#          v
#  [ Frente de Onda ] -> Expansão Esférica no Modelo Verdadeiro
#          |
#          +---> Bate no Refletor (Z=350m) ---> Gera Eco (Reflexão)
#          |
#          v
#  [ Receptores (Z=10m) ] -> Gravam o Sismograma Contínuo
# ==============================================================================

print("[Adjoint State] Rodando propagacao reversa no modelo homogeneo...")

# Isolamento de memoria e ativacao do rastreamento de gradiente na GPU
vmodel_homo = model_homo.clone().requires_grad_(True)

# ------------------------------------------------------------------------------
# Forward Pass no Modelo Homogeneo (Background)
# A execucao aciona simultaneamente a captura da onda direta (ForwardCallback) 
# e prepara os buffers de interceptacao para a onda reversa (BackwardCallback).
# ------------------------------------------------------------------------------
out_adj = scalar(
    vmodel_homo, dx, dt, max_vel=vmax_true,
    source_amplitudes=source_amplitudes,
    source_locations=source_locations,
    receiver_locations=receiver_locations,
    accuracy=8, pml_freq=peak_freq, pml_width=[30, 30, 30, 30],
    forward_callback=ForwardCallback(ishot=ctrl_shot),
    backward_callback=BackwardCallback(ishot=ctrl_shot),
    callback_frequency=callback_frequency
)
data_adj = out_adj[-1]

# ------------------------------------------------------------------------------
# Gatilho do Estado Adjunto (Backward Pass)
# 1. Calcula o residuo matricial (MSE) entre o sismograma real e o cego.
# 2. O .backward() injeta este residuo nos receptores e aciona a propagacao 
#    reversa, construindo o gradiente espacial via correlacao dos campos.
# ------------------------------------------------------------------------------
torch.nn.MSELoss()(data_true, data_adj).backward()

# ------------------------------------------------------------------------------
# Inspecao Visual da Correlacao Cruzada
# Renderiza a interacao dos campos em um instante temporal definido (snap_adj).
# ------------------------------------------------------------------------------
snap_adj = 50
vmax_fwd_s = torch.quantile(forward_snapshots[snap_adj], 0.99).item()
vmax_bwd_s = vmax_fwd_s * 1e-3  # Fator de ganho devido a atenuacao energetica
vmax_grad_s = torch.quantile(gradient_snapshots[snap_adj].abs(), 1.0).item()

fig_adj, axes_adj = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

# Campo de onda direto propagando no tempo cronologico
im0_adj = axes_adj[0].imshow(forward_snapshots[snap_adj].T, cmap="seismic")
axes_adj[0].set_title("Forward")

# Campo de residuo propagando no tempo reverso (inversao do indice: nt - snap_adj)
im1_adj = axes_adj[1].imshow(backward_snapshots[nt - snap_adj].T, cmap="seismic", vmax=vmax_bwd_s)
axes_adj[1].set_title("Backward")

# Produto cumulativo da correlacao (matriz do gradiente em formacao)
im2_adj = axes_adj[2].imshow(gradient_snapshots[nt - snap_adj].T, cmap="gray")
axes_adj[2].set_title("Gradient")

for ax in axes_adj:
    ax.set_xticks([])
    ax.set_yticks([])

fig_adj.colorbar(im0_adj, ax=axes_adj[0], shrink=0.38)
fig_adj.colorbar(im1_adj, ax=axes_adj[1], shrink=0.38)
fig_adj.colorbar(im2_adj, ax=axes_adj[2], shrink=0.38)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELULA 10: ANIMACAO HTML FINAL (FORWARD, BACKWARD, GRADIENT)
# ==============================================================================
# DIAGRAMA DE SINCRONIZAÇÃO TEMPORAL (RENDERIZAÇÃO HTML5)
# ------------------------------------------------------------------------------
#  Eixo do Tempo da Animação (Frame 0 -> Frame N)
#
#  Buffer Forward  : Lendo de Trás para Frente [N, N-1, ..., 0]
#  Buffer Backward : Lendo de Frente para Trás [0, 1, ..., N]
#  Buffer Gradient : Lendo de Frente para Trás [0, 1, ..., N] (Acumulativo)
#
#  Resultado Visual: A onda direta "volta" para a fonte enquanto o resíduo
#  "desce" dos receptores. Onde elas colidem, o refletor acende.
# ==============================================================================

print("[Animacao Final] Compilando video do Estado Adjunto completo...")

# ------------------------------------------------------------------------------
# Otimizacao de Renderizacao (Subsampling)
# Para evitar estouro de memoria no DOM do navegador (HTML5), a animacao pula 
# de 10 em 10 frames. A logica garante que o ultimo frame absoluto seja incluido.
# ------------------------------------------------------------------------------
frame_step = 10
interval_ms = 120
nframes_total = min(len(forward_snapshots), len(backward_snapshots), len(gradient_snapshots))

frames = list(range(0, nframes_total, frame_step))
if frames[-1] != nframes_total - 1:
    frames.append(nframes_total - 1)

# ------------------------------------------------------------------------------
# Calibracao Global de Amplitude
# O quantil de 99% e calculado sobre todo o tensor espaco-temporal para fixar 
# os limites do colormap (vmin/vmax). Isso impede que a escala de cores oscile 
# descontroladamente frame a frame.
# ------------------------------------------------------------------------------
vmax_fwd_all = torch.quantile(forward_snapshots, 0.99).item()
vmax_bwd_all = vmax_fwd_all * 1e-3
vmax_grad_all = torch.quantile(gradient_snapshots.abs(), 0.99).item()

fig_all, axes_all = plt.subplots(1, 3, figsize=(9, 3), sharey=True)

# Transposicao inicial (.T) para ancorar o espaco (Z, Y) na tela
im0_all = axes_all[0].imshow(forward_snapshots[0].T, cmap="seismic", vmin=-vmax_fwd_all, vmax=vmax_fwd_all, animated=True)
im1_all = axes_all[1].imshow(backward_snapshots[0].T, cmap="seismic", vmin=-vmax_bwd_all, vmax=vmax_bwd_all, animated=True)
im2_all = axes_all[2].imshow(gradient_snapshots[0].T, cmap="gray", vmin=-vmax_grad_all, vmax=vmax_grad_all, animated=True)

for ax in axes_all:
    ax.set_xticks([])
    ax.set_yticks([])

titles_all = [
    axes_all[0].set_title("Forward"),
    axes_all[1].set_title("Backward"),
    axes_all[2].set_title("Gradient"),
]

fig_all.colorbar(im0_all, ax=axes_all[0], shrink=0.8)
fig_all.colorbar(im1_all, ax=axes_all[1], shrink=0.8)
fig_all.colorbar(im2_all, ax=axes_all[2], shrink=0.8)
plt.tight_layout()
plt.close(fig_all) # Suprime a plotagem do grid estatico vazio

# ------------------------------------------------------------------------------
# Motor de Sincronizacao Temporal (O Coracao do Metodo Adjunto)
# O loop de animacao itera no referencial estrito do TEMPO REVERSO:
# - A onda direta (Forward) e acessada de tras para frente (nframes_total - 1 - frame)
# - O residuo (Backward) e o Gradiente sao acessados linearmente, pois o motor 
#   Autograd ja preencheu estes buffers em ordem retrograda durante o loss.backward().
# A convergencia fisica das ondas na tela ilumina os refletores.
# ------------------------------------------------------------------------------
def update_all(frame):
    im0_all.set_data(forward_snapshots[nframes_total - 1 - frame].T)
    im1_all.set_data(backward_snapshots[frame].T)
    im2_all.set_data(gradient_snapshots[frame].T)

    # Calculo do timestamp fisico retrocedendo
    t = (nframes_total - 1 - frame) * callback_frequency * dt
    titles_all[0].set_text(f"Forward (t = {t:.3f} s)")
    titles_all[1].set_text(f"Backward (t = {t:.3f} s)")
    titles_all[2].set_text(f"Gradient (t = {t:.3f} s)")
    
    return (im0_all, im1_all, im2_all, *titles_all)

# Compilacao em JavaScript/HTML5 para execucao interativa dentro do Jupyter
ani_all = animation.FuncAnimation(fig_all, update_all, frames=frames, interval=interval_ms, blit=True)
display(HTML(ani_all.to_jshtml()))

In [ ]:
# ==============================================================================
# CELULA 11: TRANSICAO PARA PINNs - DADOS DE REFERENCIA
# ==============================================================================
# DIAGRAMA TOPOLÓGICO: DATASET OPENFWI (FLATVEL_A)
# ------------------------------------------------------------------------------
#  Dimensões Físicas: 700m x 700m (NX=70, NZ=70, DX=10m)
#
#  [ Superfície Z=10m ]
#  Fontes (5 Tiros) :  *       *       *       *       *  (Esparsas / Few-Shot)
#  Receptores (70)  : vvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvvv (Cobertura Total)
#
#  [ Subsuperfície ]
#  Camadas Planas (FlatVel) com velocidades variando entre 1500 e 3000 m/s
# ==============================================================================
# 
# FONTE DE REFERENCIA PRIMARIA:
# As especificacoes abaixo provem do artigo cientifico do benchmark OpenFWI:
# "OpenFWI: Large-Scale Multi-Structural Benchmark Datasets for Seismic Full-Waveform Inversion"
# (Deng et al., 2022 - arXiv:2111.02926).
# Especificamente, os dados referem-se ao conjunto de dados "FlatVel_A" (Modelo 14).
#
# VERIFICACAO DE SHAPES (Metadados Fisicos vs. Estrutura do Arquivo .npy):
# - velocity_map shape: [500, 1, 70, 70] -> Indica 500 modelos, 1 canal, 70x70 de grade.
# - seismic_data shape: [500, 5, 1000, 70] -> Indica 500 simulacoes, 5 tiros, 1000 passos de tempo, 70 receptores.
# ------------------------------------------------------------------------------

import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset

print(f"[MLOps] Dispositivo ativo para treinamento da PINN: {device}")

# ------------------------------------------------------------------------------
# Parametros Geometricos e Espaciais (Padrao OpenFWI)
# Define a topologia matricial de entrada/saida para a arquitetura de Deep Learning.
# ------------------------------------------------------------------------------
DX = 10.0          # Espacamento espacial da malha (metros). Dominio total de 700m x 700m.
NX = 70            # Dimensao horizontal do grid (pontos no eixo X).
NZ = 70            # Dimensao vertical do grid (pontos no eixo Z).

# ------------------------------------------------------------------------------
# Parametros Temporais (Resolucao do Sismograma)
# Define a dimensao vetorial dos dados de pressao acustica (features da rede).
# ------------------------------------------------------------------------------
DT = 1e-3          # Discretizacao temporal rigorosa (0.001 segundos ou 1 milissegundo).
NT = 1000          # Numero de passos de tempo (janela de registro de 1.0 segundo completo = 1000 ms).

# ------------------------------------------------------------------------------
# Topologia de Aquisicao (Cobertura de Iluminacao)
# O dataset FlatVel_A impoe um problema de few-shot (apenas 5 tiros), exigindo 
# que a PINN atue como um regularizador forte para preencher as falhas de iluminacao.
# ------------------------------------------------------------------------------
NUM_SHOTS = 5      # Numero total de tiros (fontes dispersas pela superficie).
NUM_REC = 70       # Receptores gravando o campo de onda (cobertura total de topo).

print(f"Configuracao OpenFWI -> Tiros: {NUM_SHOTS} | Receptores: {NUM_REC} | Janela de Tempo: {NT} amostras")

In [ ]:
# ==============================================================================
# CELULA 12: INGESTAO DO DATASET OPENFWI EM FORMATO NUMPY (PRE-PROCESSAMENTO)
# ==============================================================================
# DIAGRAMA DE ENGENHARIA DE DADOS: INGESTÃO E HIGIENIZAÇÃO (OOM PREVENTION)
# ------------------------------------------------------------------------------
#  [ Disco (Arquivos .npy) ]
#          |
#          v
#  [ Memória RAM (Matrizes Gigantes) ] -> raw_seismic_data [500, 5, 1000, 70]
#          |
#          +---> Fatiamento (Slicing) + .copy() (Isolamento de Ponteiro)
#          |
#          v
#  [ Memória RAM (Amostra Isolada) ] ---> seismic_obs [5, 1000, 70]
#          |
#          +---> Comando 'del' nas Matrizes Gigantes
#          |
#          v
#  [ Garbage Collector (Python) ] ------> Libera Gigabytes de RAM
# ==============================================================================

# Caminhos relativos para o montante de dados pre-processados
path_seismic = '../data/FlatVel_A/FlatVel_A_data14.npy'
path_model   = '../data/FlatVel_A/FlatVel_A_model14.npy'

# ------------------------------------------------------------------------------
# Verificacao de Integridade (Sanity Check)
# Garante que o pipeline falhe rapido (Fail-Fast) caso o dataset nao esteja
# montado na arvore de diretorios esperada, evitando falhas silenciosas.
# ------------------------------------------------------------------------------
if not os.path.exists(path_seismic) or not os.path.exists(path_model):
    print("[ALERTA CRITICO] Arquivos nao encontrados! Verifique o diretorio '../data/FlatVel_A/'.")
else:
    print("[Data Ingestion] Arquivos localizados. Carregando tensores originais...")

    # Carregamento do binario NumPy diretamente para a memoria RAM
    raw_seismic_data = np.load(path_seismic)
    raw_velocity_map = np.load(path_model)

    print(f"Shape original dos Sismogramas: {raw_seismic_data.shape}")
    print(f"Shape original das Velocidades: {raw_velocity_map.shape}")

    # Isolando a primeira amostra (Indice 0) para a prova de conceito base (Baseline)
    SAMPLE_INDEX = 0

    # --------------------------------------------------------------------------
    # Fatiamento Seguro (Memory Decoupling)
    # O uso obrigatorio do .copy() forca a alocacao de um novo bloco de memoria.
    # Sem isso, o fatiamento retorna apenas uma "view" (ponteiro), o que 
    # impediria a liberacao da matriz original massiva pelo Garbage Collector.
    # --------------------------------------------------------------------------
    seismic_obs = raw_seismic_data[SAMPLE_INDEX, :, :, :].copy()
    true_velocity = raw_velocity_map[SAMPLE_INDEX, 0, :, :].copy() 

    print("\n[Data Engineering] Dados fatiados para o pipeline PINN:")
    print(f"Sismograma de Estudo (seismic_obs): {seismic_obs.shape}")
    print(f"Velocidade de Estudo (true_velocity): {true_velocity.shape}")
    
    # --------------------------------------------------------------------------
    # Higienizacao de Memoria (OOM Prevention)
    # A exclusao explicita das variaveis e vital para ambientes com restricao 
    # de RAM, liberando o ponteiro para o Python Garbage Collector limpar o heap.
    # --------------------------------------------------------------------------
    del raw_seismic_data
    del raw_velocity_map
    print("[Sistema] Memoria RAM higienizada. Matrizes originais descartadas.")

In [ ]:
# ==============================================================================
# CELULA 13: VISUALIZACAO EXPLORATORIA DA FISICA (SANITY CHECK)
# ==============================================================================
# DIAGRAMA DE INSPEÇÃO VISUAL (SANITY CHECK)
# ------------------------------------------------------------------------------
#  [ Matriz de Velocidade (Ground Truth) ]
#  Verifica se a geometria (70x70) bate com as coordenadas físicas.
#
#  [ Sismogramas (5 Tiros) ]
#  Verifica se a energia acústica (wavelet) está presente e se os
#  tempos de trânsito fazem sentido físico (sem ruído numérico espúrio).
# ==============================================================================
print("[Visualizacao] Desenhando Modelo Verdadeiro e Sismogramas (OpenFWI)...")

# ------------------------------------------------------------------------------
# Inspecao do Espaco Latente Fisico (Ground Truth)
# Plotagem do modelo de velocidade acustica sobreposto a topologia de aquisicao.
# Valida visualmente se a malha de tensores corresponde a geometria do benchmark
# antes do envio para a GPU.
# ------------------------------------------------------------------------------
plt.figure(figsize=(8, 5))
img_vel = plt.imshow(true_velocity, cmap='jet', aspect='auto')

# Posicionamento dos Geofones (Receptores) - Cobertura de superficie completa
rec_x = np.arange(NUM_REC)
rec_z = np.ones(NUM_REC) * 1  
plt.plot(rec_x, rec_z, 'kv', markersize=4, label='Receptores (70)')

# Posicionamento das Fontes (Tiros) - Cobertura esparsa (Few-Shot)
shot_x = np.linspace(0, NX-1, NUM_SHOTS)
shot_z = np.ones(NUM_SHOTS) * 1
plt.plot(shot_x, shot_z, 'r*', markersize=12, label='Fontes (5)')

plt.colorbar(img_vel, label='Velocidade Acustica (m/s)')
plt.title('Modelo de Velocidade 14 e Geometria de Aquisicao')
plt.xlabel('Eixo X (Indice do Grid)')
plt.ylabel('Profundidade Z (Indice do Grid)')
plt.legend(loc='upper right', bbox_to_anchor=(1.4, 1.0))
plt.show()

# ------------------------------------------------------------------------------
# QA de Dados Sismicos (Quality Assurance dos Sismogramas)
# Plotagem de todos os 5 tiros para garantir que a energia esta varrendo 
# o grid geologico inteiro. O uso do laco 'for' mantem a rastreabilidade algoritmica.
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, NUM_SHOTS, figsize=(20, 5), sharey=True)

for i in range(NUM_SHOTS):
    shot_data = seismic_obs[i, :, :]
    axes[i].imshow(shot_data, cmap='gray', aspect='auto', vmin=-0.5, vmax=0.5)
    
    # Desenha a linha de geofones
    axes[i].plot(rec_x, np.zeros(NUM_REC), 'kv', markersize=3, alpha=0.5)
    
    # Calcula a coordenada X exata da fonte ativa atual para posicionar a estrela
    active_src_x = shot_x[i]
    
    # Destaca a fonte ativa
    axes[i].plot(active_src_x, 0, 'r*', markersize=15, label=f'Fonte Ativa') 
    axes[i].set_title(f'Tiro {i}')
    axes[i].set_xlabel('Geofones (Receptores)')
    
    # Ajusta a legenda para nao cobrir o sinal (esquerda para tiros a direita e vice-versa)
    loc = 'upper right' if i < (NUM_SHOTS / 2) else 'upper left'
    axes[i].legend(loc=loc)

# Apenas o primeiro eixo precisa do rotulo Y (Tempo) para nao poluir o grafico
axes[0].set_ylabel('Tempo (Amostras)')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELULA 14: ENGENHARIA DE DADOS - NUVEM DE PONTOS E GEOMETRIA DAS FONTES
# ==============================================================================
# DIAGRAMA DE TRANSFORMAÇÃO: HIPERCUBO -> NUVEM DE PONTOS CONTÍNUA
# ------------------------------------------------------------------------------
#  [ Hipercubo Discreto (NumPy) ]
#  Dimensões: (T=1000, Z=1, X=70) -> Matriz 3D
#          |
#          v ( np.meshgrid + .flatten() )
#  [ Nuvem de Pontos (PyTorch Tensors) ]
#  Coluna X: [x0, x1, x2, ..., xN]
#  Coluna Z: [z0, z1, z2, ..., zN]
#  Coluna T: [t0, t1, t2, ..., tN]
#          |
#          v ( Min-Max Scaling )
#  [ Domínio Topológico Normalizado ] -> Todas as coordenadas entre [-1, 1]
# ==============================================================================
print("[Data Engineering] Iniciando transformacao: Grid NumPy -> Nuvem de Pontos PyTorch...")

# ------------------------------------------------------------------------------
# Mapeamento Fisico das Fontes Sismicas (Geometria de Aquisicao)
# Calcula as coordenadas exatas (X, Z) em metros para a injecao da energia 
# da fonte na Equacao Diferencial Parcial (PDE). Fundamental para a PINN.
# ------------------------------------------------------------------------------
espacamento_tiros = (NX - 1) * DX / (NUM_SHOTS - 1)
posicoes_fontes = [] 

for i in range(NUM_SHOTS):
    x_s = i * espacamento_tiros
    z_s = 10.0  # Profundidade constante da fonte (10 metros)
    posicoes_fontes.append((x_s, z_s))

print(f"Coordenadas das {NUM_SHOTS} fontes calculadas: {posicoes_fontes}")

# ------------------------------------------------------------------------------
# Estruturacao do Dataset para a PINN (Continuous Point Cloud)
# A QUEBRA DE PARADIGMA: Redes Neurais Informadas pela Fisica (PINNs) nao 
# consomem "imagens" ou matrizes como as CNNs tradicionais. 
# Elas aprendem uma FUNCAO CONTINUA: f(x, z, t) = amplitude.
# Por isso, precisamos "quebrar" (flatten) a matriz do sismograma original (*.npy) 
# em uma lista gigante de pontos independentes (uma nuvem de pontos).
# ------------------------------------------------------------------------------
class SeismicDataDataset(Dataset):
    """
    Motor de Traducao NumPy -> PyTorch.
    Converte hipercubos discretos em tensores de coordenadas continuas.
    """
    def __init__(self, d_obs: np.ndarray):
        super().__init__()
        
        # 1. Vetores Fisicos de Discretizacao (Ainda no ecossistema NumPy)
        x_rec = np.arange(NUM_REC) * DX
        z_rec = np.array([10.0]) # Receptores fixos em 10m de profundidade
        t_vec = np.arange(NT) * DT
        
        # 2. Malha Continua (Meshgrid)
        # Cria todas as combinacoes possiveis de Tempo, Profundidade e Distancia
        T, Z, X = np.meshgrid(t_vec, z_rec, x_rec, indexing='ij')
        
        # 3. Handoff NumPy -> PyTorch (Conversao para Tensores 1D)
        # Aqui achatamos a malha (.flatten). Em vez de uma matriz, agora temos
        # colunas gigantes. O 'torch.tensor' acopla os dados no ecossistema do 
        # PyTorch, e o 'float32' e a precisao padrao exigida pelas GPUs.
        self.t_data = torch.tensor(T.flatten(), dtype=torch.float32)
        self.z_data = torch.tensor(Z.flatten(), dtype=torch.float32)
        self.x_data = torch.tensor(X.flatten(), dtype=torch.float32)
        
        # 4. Target (Amplitudes Sismicas / Labels da Rede)
        # Para cada coordenada (x, z, t) extraida acima, qual e a pressao acustica 
        # real registrada no sismograma? Agrupamos os 5 tiros lado a lado.
        u_reshaped = np.zeros((len(self.t_data), NUM_SHOTS))
        for i in range(NUM_SHOTS):
            u_reshaped[:, i] = d_obs[i, :, :].flatten()
            
        # Converte o bloco de respostas (targets) para Tensor PyTorch
        self.u_data = torch.tensor(u_reshaped, dtype=torch.float32)

        # 5. Normalizacao Topologica [-1, 1] (Evitando Explosao de Gradientes)
        # O espaco (X) vai ate 700 metros. O tempo (T) vai ate 1.0 segundo.
        # Se entregarmos essas escalas tao diferentes para a Rede Neural, os pesos 
        # vao distorcer e a rede nao aprendera. O Min-Max scaling equaliza tudo.
        self.x_norm = self.normalize(self.x_data, 0.0, (NX-1)*DX)
        self.z_norm = self.normalize(self.z_data, 0.0, (NZ-1)*DX)
        self.t_norm = self.normalize(self.t_data, 0.0, (NT-1)*DT)

    def normalize(self, tensor: torch.Tensor, min_val: float, max_val: float) -> torch.Tensor:
        """Aplica escalonamento Min-Max estrito, espremendo os dados entre -1 e 1."""
        return 2.0 * ((tensor - min_val) / (max_val - min_val)) - 1.0

    def __len__(self) -> int:
        """Define o limite de iteracao: o numero total de coordenadas no dominio."""
        return len(self.x_data)

    def __getitem__(self, idx: int) -> Tuple[Tuple[torch.Tensor, torch.Tensor, torch.Tensor], torch.Tensor]:
        """
        O Motor de Injecao do DataLoader: 
        A cada iteracao de treinamento, o PyTorch pede "me de a proxima linha".
        Retornamos a tupla de coordenadas normalizadas (X, Z, T) e a amplitude real associada.
        """
        return (self.x_norm[idx], self.z_norm[idx], self.t_norm[idx]), self.u_data[idx]

# Instanciacao do dataset na memoria RAM
dataset_obs = SeismicDataDataset(seismic_obs)
print(f"Volume do Dataset: {len(dataset_obs)} coordenadas espaco-temporais geradas.")

In [ ]:
# ==============================================================================
# CELULA 15: ARQUITETURA DA REDE NEURAL (MULTI-LAYER PERCEPTRON - MLP)
# ==============================================================================
# DIAGRAMA DE ARQUITETURA: PINN MULTI-SOURCE (MLP)
# ------------------------------------------------------------------------------
#  [ Entrada Contínua ]   [ Camadas Ocultas (6x128) ]     [ Saída (Amplitudes) ]
#  (Coordenadas Norm.)    (Ativação: Tanh - Suave)        (Pressão Acústica)
#
#       X_norm --+                                        +--> U_shot_1
#                |        +-----+   +-----+       +-----+ |--> U_shot_2
#       Z_norm --+------> | 128 |-->| 128 |-->...>| 128 |-+--> U_shot_3
#                |        +-----+   +-----+       +-----+ |--> U_shot_4
#       T_norm --+                                        +--> U_shot_5
#
#  *Obrigatório: Tanh garante derivadas de 2ª ordem não nulas (Laplaciano).*
# ==============================================================================

import torch
import torch.nn as nn

class PINN_MLP(nn.Module):
    """
    O QUE FAZ:
    Mapeia coordenadas contínuas (X, Z, T) para amplitudes sísmicas (U1...U5).
    
    PARA QUE SERVE:
    Atua como o simulador substituto da Equação da Onda. Ao forçar a rede a prever 
    os 5 tiros simultaneamente, garantimos que ela aprenda uma única geologia 
    consistente, atuando como um forte regularizador físico.
    """
    def __init__(self, in_features=3, out_features=5, hidden_layers=6, hidden_neurons=128):
        super().__init__() 
        self.layers = nn.ModuleList()
        
        # Camada de Entrada
        self.layers.append(nn.Linear(in_features, hidden_neurons))
        
        # Camadas Ocultas
        for _ in range(hidden_layers):
            self.layers.append(nn.Linear(hidden_neurons, hidden_neurons))
            
        # Camada de Saída
        self.layers.append(nn.Linear(hidden_neurons, out_features))
        
    def forward(self, x_in: torch.Tensor, z_in: torch.Tensor, t_in: torch.Tensor) -> torch.Tensor:
        # ----------------------------------------------------------------------
        # CORREÇÃO CRÍTICA DE TOPOLOGIA TENSORIAL:
        # Transforma tensores 1D (BATCH_SIZE,) em vetores coluna (BATCH_SIZE, 1).
        # Isso impede que o torch.cat crie um vetor gigante de 24576 elementos,
        # forçando a criação de uma matriz correta de formato (BATCH_SIZE, 3).
        # ----------------------------------------------------------------------
        x_col = x_in.view(-1, 1)
        z_col = z_in.view(-1, 1)
        t_col = t_in.view(-1, 1)
        
        # Concatena no eixo das features (dim=1)
        u = torch.cat([x_col, z_col, t_col], dim=1)
        
        # Propagação com ativação Tanh (infinitamente diferenciável)
        for i in range(len(self.layers) - 1):
            u = self.layers[i](u)
            u = torch.tanh(u)
            
        # Saída linear
        u = self.layers[-1](u)
        return u

print("[Arquitetura] Instanciando a PINN (Physics-Informed Neural Network)...")

# Instanciação da rede com os parâmetros rigorosos do setup few-shot (5 tiros)
model_pinn = PINN_MLP(in_features=3, out_features=NUM_SHOTS, hidden_layers=6, hidden_neurons=128)

# Transferência para a GPU
model_pinn = model_pinn.to(device)
print(model_pinn)

In [ ]:
# ==============================================================================
# CELULA 16: MOTOR DA FISICA (AUTOGRAD + CORRECAO JACOBIANA)
# ==============================================================================
# DIAGRAMA DE FLUXO: PINN + Autograd + Regra da Cadeia
# ------------------------------------------------------------------------------
#  [ Predição da Rede (u_pred) ]
#          |
#          +---> (Autograd: Derivadas no Domínio Normalizado [-1, 1])
#          |      u_xx_norm, u_zz_norm, u_tt_norm
#          |
#          v
#  [ Correção Jacobiana (Regra da Cadeia) ]
#  Multiplica pelos fatores de escala (2 / (Max - Min))^2
#          |
#          v
#  [ Derivadas no Domínio Físico Real (Metros e Segundos) ]
#  u_xx_phys, u_zz_phys, u_tt_phys
#          |
#          v
#  [ Resíduo da PDE (Equação da Onda) ] -> u_tt - c^2 * (u_xx + u_zz) = 0
# ==============================================================================
import torch
import torch.nn as nn
from typing import Tuple

def get_gradient(output: torch.Tensor, input_var: torch.Tensor) -> torch.Tensor:
    """
    O QUE FAZ:
    Calcula a derivada parcial exata de um tensor de saída em relação a um tensor de entrada
    utilizando o Histórico de Operações da Regra da Cadeia (Autograd).

    PARA QUE SERVE:
    Extrair as taxas de variação da onda. Na geofísica, precisamos saber como a pressão 
    acústica muda no espaço (inclinação/curvatura) e no tempo (velocidade/aceleração) 
    para verificar se a rede neural está obedecendo às leis da termodinâmica e mecânica.
    """
    grad_outputs = torch.ones_like(output)
    gradient = torch.autograd.grad(
        outputs=output,
        inputs=input_var,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True
    )[0] 
    return gradient

def compute_physics_loss(
    model: nn.Module, 
    x_norm: torch.Tensor, 
    z_norm: torch.Tensor, 
    t_norm: torch.Tensor, 
    c_velocity: torch.Tensor,
    scale_factors: dict
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    O QUE FAZ:
    Calcula o resíduo da Equação Diferencial Parcial (PDE) da onda acústica 2D.
    Aplica os fatores de escala (Jacobianos) para reverter a normalização [-1, 1] 
    durante o cálculo das derivadas, garantindo a integridade dimensional.

    PARA QUE SERVE:
    Atua como o "Gabarito da Natureza". Força a rede neural a respeitar a física 
    da propagação sísmica. Se a rede "alucinar" dados que violam a equação da onda, 
    esta função gera uma penalidade (Loss) alta, forçando o otimizador a corrigir os pesos.
    """
    # Ativação do rastreamento de gradientes
    x_norm.requires_grad_(True)
    z_norm.requires_grad_(True)
    t_norm.requires_grad_(True)
    
    # Forward Pass: Predição do campo de onda
    u_pred = model(x_norm, z_norm, t_norm) 
    
    # --------------------------------------------------------------------------
    # 1. Derivadas no Domínio Normalizado (Autograd Puro)
    # --------------------------------------------------------------------------
    u_x_norm = get_gradient(u_pred, x_norm)
    u_xx_norm = get_gradient(u_x_norm, x_norm)
    
    u_z_norm = get_gradient(u_pred, z_norm)
    u_zz_norm = get_gradient(u_z_norm, z_norm)
    
    u_t_norm = get_gradient(u_pred, t_norm)
    u_tt_norm = get_gradient(u_t_norm, t_norm)
    
    # --------------------------------------------------------------------------
    # 2. Correção Jacobiana (Retorno ao Domínio Físico)
    # Multiplicamos pelo quadrado do fator de escala pois são derivadas de 2ª ordem.
    # --------------------------------------------------------------------------
    j_x = scale_factors['x']  # (2.0 / (X_max - X_min))
    j_z = scale_factors['z']  # (2.0 / (Z_max - Z_min))
    j_t = scale_factors['t']  # (2.0 / (T_max - T_min))
    
    u_xx_phys = u_xx_norm * (j_x ** 2)
    u_zz_phys = u_zz_norm * (j_z ** 2)
    u_tt_phys = u_tt_norm * (j_t ** 2)
    
    # --------------------------------------------------------------------------
    # 3. O Teste da Natureza (Equação da Onda Acústica 2D)
    # --------------------------------------------------------------------------
    # Resíduo = Aceleração - Velocidade^2 * (Curvatura X + Curvatura Z)
    pde_residual = u_tt_phys - (c_velocity ** 2) * (u_xx_phys + u_zz_phys)
    
    # Erro Quadrático Médio da violação física
    loss_pde = torch.mean(pde_residual ** 2)
    
    return loss_pde, u_pred

In [ ]:
# ==============================================================================
# CELULA 17: MOTOR DE TREINAMENTO MLOPS E INVERSÃO FWI
# ==============================================================================
# DIAGRAMA DE FLUXO MLOps: MOTOR DE TREINAMENTO DELTA-PINN (EPOCH LOOP)
# ------------------------------------------------------------------------------
#  [ DataLoader (Stochastic Batching) ] ---> Alimenta a GPU em Lotes (Batches)
#                 |
#                 v
#  +------------------------------------------------------------------------+
#  | 1. Forward Pass (Rede Neural) -> u_pred (Amplitudes Sintéticas)        |
#  | 2. Interpolação Geológica -> Extrai c_velocity do VelocityGrid         |
#  | 3. Data Loss -> MSE(u_pred, u_obs) nos Receptores                      |
#  | 4. Physics Loss -> compute_physics_loss() (Resíduo da Equação da Onda) |
#  | 5. Loss Total -> (λ_data * Data Loss) + (λ_pde * Physics Loss)         |
#  | 6. Backward Pass -> loss.backward() (Autograd calcula os gradientes)   |
#  | 7. Optimizer Step -> Atualiza Pesos da Rede E o Modelo de Velocidade   |
#  +------------------------------------------------------------------------+
#                 |
#                 v
#  [ Logger / MLflow ] ---> Registra Métricas e Limpa VRAM (empty_cache)
# ==============================================================================

import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

class VelocityGrid(nn.Module):
    """
    O QUE FAZ: 
    Mantém a matriz de velocidades (o mapa da subsuperfície) como parâmetros 
    treináveis na GPU. Utiliza interpolação bilinear (grid_sample) para fornecer 
    a velocidade exata em qualquer coordenada contínua (x, z) solicitada pela PINN.

    PARA QUE SERVE: 
    Este é o PRODUTO COMERCIAL (O Santo Graal). Enquanto a PINN atua como o 
    simulador da onda, esta matriz é a geologia real sendo descoberta. Ao final 
    do treinamento, a PINN é descartada e este grid é entregue ao cliente.
    """
    def __init__(self, nx: int, nz: int, initial_vel: float):
        super().__init__()
        # Formato exigido pelo grid_sample: [Batch, Canais, Altura, Largura]
        # Inicializamos com o modelo cego (background) de 1500 m/s
        self.grid = nn.Parameter(torch.ones(1, 1, nz, nx) * initial_vel)
        
    def forward(self, x_norm: torch.Tensor, z_norm: torch.Tensor) -> torch.Tensor:
        # Agrupa as coordenadas [-1, 1] no formato [1, Batch, 1, 2] para o amostrador
        grid_coords = torch.cat([x_norm.unsqueeze(-1), z_norm.unsqueeze(-1)], dim=-1)
        grid_coords = grid_coords.view(1, -1, 1, 2)
        
        # Extrai a velocidade interpolada para cada ponto da nuvem
        c = F.grid_sample(self.grid, grid_coords, align_corners=True)
        return c.view(-1) # Retorna como vetor 1D alinhado com a nuvem de pontos

print(f"[MLOps] Dispositivo ativo para treinamento da PINN: {device}")
print("[MLOps] Inicializando Motor de Treinamento...")

# ------------------------------------------------------------------------------
# 1. Configuração de Hiperparâmetros e Hardware
# ------------------------------------------------------------------------------
BATCH_SIZE = 8192       # Lotes grandes maximizam o uso dos Tensor Cores da GPU
EPOCHS = 100            # Número de varreduras completas no dataset
LR_PINN = 1e-3          # Taxa de aprendizado para a Rede Neural
LR_VEL = 10.0           # Taxa de aprendizado agressiva para o Grid de Velocidade (metros/s)

LAMBDA_DATA = 1.0       # Peso da fidelidade aos dados sísmicos
LAMBDA_PDE = 0.01       # Peso do regularizador físico (Equação da Onda)

# ------------------------------------------------------------------------------
# 2. Instanciação de Componentes (Data, Modelos e Otimizador)
# ------------------------------------------------------------------------------
# DataLoader com pin_memory=True acelera a transferência RAM -> VRAM
dataloader = DataLoader(dataset_obs, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

# Instancia o Produto Comercial (Grid de Velocidade) na GPU
vel_model = VelocityGrid(nx=NX, nz=NZ, initial_vel=1500.0).to(device)

# Otimizador unificado: Treina a Rede (Física) e o Grid (Geologia) simultaneamente
optimizer = torch.optim.Adam([
    {'params': model_pinn.parameters(), 'lr': LR_PINN},
    {'params': vel_model.parameters(), 'lr': LR_VEL}
])

# ------------------------------------------------------------------------------
# 3. Cálculo Estrito dos Fatores Jacobianos (Regra da Cadeia)
# ------------------------------------------------------------------------------
# Fator = 2.0 / (Max - Min). Necessário para reverter a normalização [-1, 1]
scale_factors = {
    'x': 2.0 / ((NX - 1) * DX),
    'z': 2.0 / ((NZ - 1) * DX),
    't': 2.0 / ((NT - 1) * DT)
}

# ------------------------------------------------------------------------------
# 4. Loop de Treinamento (Epoch Loop)
# ------------------------------------------------------------------------------
print(f"[Treinamento] Iniciando otimização conjunta (PINN + FWI) por {EPOCHS} épocas.")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_data_loss = 0.0
    epoch_pde_loss = 0.0
    
    # Modo de treinamento ativado
    model_pinn.train()
    
    for batch_idx, (coords, u_obs) in enumerate(dataloader):
        # Desempacota e envia para a GPU
        x_b, z_b, t_b = coords[0].to(device), coords[1].to(device), coords[2].to(device)
        u_obs = u_obs.to(device)
        
        optimizer.zero_grad()
        
        # --- A. Extração da Geologia ---
        # A rede consulta qual é a velocidade atual naquelas coordenadas
        c_batch = vel_model(x_b, z_b)
        
        # --- B. O Teste da Natureza (Physics Loss) ---
        # A função refatorada na Célula 16 calcula o resíduo da PDE e retorna a predição
        loss_pde, u_pred = compute_physics_loss(
            model_pinn, x_b, z_b, t_b, c_batch, scale_factors
        )
        
        # --- C. O Teste dos Dados (Data Loss) ---
        # Compara a amplitude prevista com o sismograma real gravado pelos geofones
        loss_data = torch.nn.MSELoss()(u_pred, u_obs)
        
        # --- D. Função Objetivo Global e Retropropagação ---
        loss_total = (LAMBDA_DATA * loss_data) + (LAMBDA_PDE * loss_pde)
        loss_total.backward()
        
        # Condicionamento de Gradiente (Evita explosão numérica)
        torch.nn.utils.clip_grad_norm_(model_pinn.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Acumula métricas para o Logger (usando .item() para não vazar memória)
        epoch_data_loss += loss_data.item()
        epoch_pde_loss += loss_pde.item()
        
    # --------------------------------------------------------------------------
    # 5. MLOps Logging e Higienização de Memória
    # --------------------------------------------------------------------------
    avg_data_loss = epoch_data_loss / len(dataloader)
    avg_pde_loss = epoch_pde_loss / len(dataloader)
    
    # Monitoramento da sanidade geológica (Velocidade Min/Max)
    with torch.no_grad():
        v_min = vel_model.grid.min().item()
        v_max = vel_model.grid.max().item()
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:03d}/{EPOCHS}] | "
              f"Data Loss: {avg_data_loss:.4e} | "
              f"PDE Loss: {avg_pde_loss:.4e} | "
              f"V_min: {v_min:.1f} m/s | V_max: {v_max:.1f} m/s")
        
    # Limpeza proativa do cache da GPU (Prevenção de OOM em HPC)
    torch.cuda.empty_cache()

elapsed = time.time() - start_time
print(f"[MLOps] Treinamento concluído em {elapsed/60:.2f} minutos.")

In [ ]:
# ==============================================================================
# CELULA 18: DASHBOARD DE INFERÊNCIA E QUALITY ASSURANCE (QA)
# ==============================================================================
# DIAGRAMA DE INFERÊNCIA E QA (QUALITY ASSURANCE)
# ------------------------------------------------------------------------------
#  [ GPU VRAM ]
#       |
#       +---> vel_model.grid.detach().cpu() ---> [ Matriz Invertida (2D) ]
#                                                       |
#  [ Memória RAM ]                                      v
#       |                                     [ Comparação Forense ]
#       +---> true_velocity (Ground Truth) -------->    |
#                                                       v
#                                             +--------------------+
#                                             | 1. Modelo Real     |
#                                             | 2. Modelo PINN     |
#                                             | 3. Mapa de Erro    |
#                                             | 4. Perfil 1D (Poço)|
#                                             +--------------------+
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt

def generate_qa_dashboard(vel_model: torch.nn.Module, true_vel: np.ndarray):
    """
    O QUE FAZ:
    Extrai o grid de velocidades treinado da GPU, converte para NumPy e plota 
    um painel comparativo com 4 visões: Ground Truth, Predição, Erro Absoluto 
    e um Perfil de Poço 1D (Trace).

    PARA QUE SERVE:
    Auditoria visual do produto final. Na indústria, não entregamos "Losses" 
    para o cliente, entregamos imagens da subsuperfície. O Perfil 1D simula 
    a perfuração de um poço exploratório no centro do modelo para verificar 
    se a PINN acertou a profundidade exata das camadas geológicas.
    """
    print("[Inferência] Extraindo o Produto Comercial da GPU...")
    
    # 1. Extração Segura (Descolamento do Grafo Computacional)
    # .detach() corta a ligação com o Autograd.
    # .cpu() move da VRAM para a RAM.
    # .squeeze() remove as dimensões de Batch e Canal [1, 1, NZ, NX] -> [NZ, NX]
    inverted_vel = vel_model.grid.detach().cpu().squeeze().numpy()
    
    # 2. Cálculo do Erro Absoluto
    error_map = np.abs(true_vel - inverted_vel)
    
    # 3. Extração do Perfil 1D (Simulação de Poço no centro do eixo X)
    center_x = NX // 2
    trace_true = true_vel[:, center_x]
    trace_inv = inverted_vel[:, center_x]
    depth_axis = np.arange(NZ) * DX
    
    # 4. Renderização do Dashboard
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("DELTA-PINN ENTERPRISE: Relatório de Quality Assurance (QA)", fontsize=16, fontweight='bold')
    
    # Escala de cores unificada para os modelos de velocidade
    vmin = min(true_vel.min(), inverted_vel.min())
    vmax = max(true_vel.max(), inverted_vel.max())
    
    # --- Plot 1: Ground Truth ---
    im0 = axes[0, 0].imshow(true_vel, cmap='jet', vmin=vmin, vmax=vmax, aspect='auto')
    axes[0, 0].set_title("Modelo Verdadeiro (Ground Truth)")
    axes[0, 0].set_ylabel("Profundidade (Z)")
    fig.colorbar(im0, ax=axes[0, 0], label="Velocidade (m/s)")
    
    # --- Plot 2: Modelo Invertido (PINN) ---
    im1 = axes[0, 1].imshow(inverted_vel, cmap='jet', vmin=vmin, vmax=vmax, aspect='auto')
    axes[0, 1].set_title(f"Modelo Invertido (PINN - {EPOCHS} Épocas)")
    fig.colorbar(im1, ax=axes[0, 1], label="Velocidade (m/s)")
    
    # --- Plot 3: Mapa de Erro Absoluto ---
    im2 = axes[1, 0].imshow(error_map, cmap='magma', aspect='auto')
    axes[1, 0].set_title("Mapa de Erro Absoluto |True - PINN|")
    axes[1, 0].set_xlabel("Distância (X)")
    axes[1, 0].set_ylabel("Profundidade (Z)")
    fig.colorbar(im2, ax=axes[1, 0], label="Erro (m/s)")
    
    # --- Plot 4: Perfil de Poço 1D ---
    axes[1, 1].plot(trace_true, depth_axis, 'k-', linewidth=2, label="Perfil Real")
    axes[1, 1].plot(trace_inv, depth_axis, 'r--', linewidth=2, label="Perfil PINN")
    axes[1, 1].invert_yaxis() # Profundidade cresce para baixo
    axes[1, 1].set_title(f"Perfil de Poço 1D (X = {center_x * DX} m)")
    axes[1, 1].set_xlabel("Velocidade (m/s)")
    axes[1, 1].set_ylabel("Profundidade (m)")
    axes[1, 1].legend()
    axes[1, 1].grid(True, linestyle=':', alpha=0.7)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

# Executa a geração do Dashboard
generate_qa_dashboard(vel_model, true_velocity)

In [ ]:
# ==============================================================================
# CELULA 19 [FAST-TRACK]: MOTOR DE CONVERGÊNCIA PROFUNDA (ADAM -> L-BFGS)
# ==============================================================================
# DIAGRAMA DE FLUXO HPC: FAST-TRACK L-BFGS
# ------------------------------------------------------------------------------
#  [ ESTÁGIO 1: ADAM (Pré-Treino Rápido) ] -> Apenas 500 Épocas
#  Objetivo: Tirar a PINN do caos inicial. Interrompido antes da "morte clínica".
#          |
#          v 
#  [ ESTÁGIO 2: L-BFGS (O Motor Principal) ] -> 1000 Épocas
#  Objetivo: Usar a Matriz Hessiana para perfurar a barreira dos 1500 m/s e 
#  esculpir a geologia profunda.
# ==============================================================================

import time
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau

print("[MLOps HPC] Inicializando Motor Fast-Track (Adam -> L-BFGS)...")

# ------------------------------------------------------------------------------
# 1. Hiperparâmetros Rebalanceados (Baseado na Telemetria)
# ------------------------------------------------------------------------------
EPOCHS_ADAM = 500       # Reduzido: Evita desperdício de GPU após o colapso da LR
EPOCHS_LBFGS = 1000     # Aumentado: O verdadeiro motor da inversão geofísica
LR_PINN_ADAM = 1e-3
LR_VEL_ADAM = 10.0

# Reset limpo do modelo
model_pinn = PINN_MLP(in_features=3, out_features=NUM_SHOTS, hidden_layers=6, hidden_neurons=128).to(device)
vel_model = VelocityGrid(nx=NX, nz=NZ, initial_vel=1500.0).to(device)

# ==============================================================================
# ESTÁGIO 1: ADAM (PRÉ-TREINO DA FÍSICA)
# ==============================================================================
optimizer_adam = torch.optim.Adam([
    {'params': model_pinn.parameters(), 'lr': LR_PINN_ADAM},
    {'params': vel_model.parameters(), 'lr': LR_VEL_ADAM}
])

scheduler = ReduceLROnPlateau(optimizer_adam, mode='min', factor=0.5, patience=50)

print(f"\n>>> INICIANDO ESTÁGIO 1: ADAM ({EPOCHS_ADAM} Épocas) <<<")
start_time = time.time()

for epoch in range(1, EPOCHS_ADAM + 1):
    model_pinn.train()
    epoch_loss = 0.0
    current_lr = optimizer_adam.param_groups[0]['lr']
    
    for coords, u_obs in dataloader:
        x_b, z_b, t_b = coords[0].to(device), coords[1].to(device), coords[2].to(device)
        u_obs = u_obs.to(device)
        
        optimizer_adam.zero_grad()
        
        c_batch = vel_model(x_b, z_b)
        loss_pde, u_pred = compute_physics_loss(model_pinn, x_b, z_b, t_b, c_batch, scale_factors)
        loss_data = torch.nn.MSELoss()(u_pred, u_obs)
        
        loss_total = (LAMBDA_DATA * loss_data) + (LAMBDA_PDE * loss_pde)
        loss_total.backward()
        
        torch.nn.utils.clip_grad_norm_(model_pinn.parameters(), max_norm=1.0)
        optimizer_adam.step()
        
        epoch_loss += loss_total.item()
        
    avg_loss = epoch_loss / len(dataloader)
    scheduler.step(avg_loss) 
    
    new_lr = optimizer_adam.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"[MLOps Alerta] Platô na Época {epoch}. LR reduzida para {new_lr:.2e}")
    
    if epoch % 100 == 0 or epoch == 1:
        with torch.no_grad():
            v_min, v_max = vel_model.grid.min().item(), vel_model.grid.max().item()
        print(f"Adam Epoch [{epoch:04d}/{EPOCHS_ADAM}] | Loss: {avg_loss:.4e} | V_min: {v_min:.1f} | V_max: {v_max:.1f}")

# ==============================================================================
# ESTÁGIO 2: L-BFGS (A ESCULTURA DA GEOLOGIA)
# ==============================================================================
print(f"\n>>> INICIANDO ESTÁGIO 2: L-BFGS ({EPOCHS_LBFGS} Épocas) <<<")

optimizer_lbfgs = torch.optim.LBFGS(
    list(model_pinn.parameters()) + list(vel_model.parameters()),
    lr=1.0,
    max_iter=20,
    max_eval=25,
    tolerance_grad=1e-7,
    tolerance_change=1e-9,
    history_size=50,
    line_search_fn="strong_wolfe" 
)

lbfgs_iterator = iter(DataLoader(dataset_obs, batch_size=16384, shuffle=True))
coords_lbfgs, u_obs_lbfgs = next(lbfgs_iterator)

x_l, z_l, t_l = coords_lbfgs[0].to(device), coords_lbfgs[1].to(device), coords_lbfgs[2].to(device)
u_obs_lbfgs = u_obs_lbfgs.to(device)

for epoch in range(1, EPOCHS_LBFGS + 1):
    
    def closure():
        optimizer_lbfgs.zero_grad()
        c_l = vel_model(x_l, z_l)
        loss_pde_l, u_pred_l = compute_physics_loss(model_pinn, x_l, z_l, t_l, c_l, scale_factors)
        loss_data_l = torch.nn.MSELoss()(u_pred_l, u_obs_lbfgs)
        
        loss_total_l = (LAMBDA_DATA * loss_data_l) + (LAMBDA_PDE * loss_pde_l)
        loss_total_l.backward()
        
        # Condicionamento de Gradiente para L-BFGS (Evita explosão da Hessiana)
        torch.nn.utils.clip_grad_norm_(model_pinn.parameters(), max_norm=1.0)
        
        return loss_total_l

    optimizer_lbfgs.step(closure)
    
    if epoch % 50 == 0 or epoch == 1:
        loss_val = closure().item()
        with torch.no_grad():
            # Força a velocidade a ficar dentro de limites geológicos realistas (Água/Sedimento -> Rocha Dura)
            vel_model.grid.clamp_(min=1400.0, max=4500.0)
            v_min, v_max = vel_model.grid.min().item(), vel_model.grid.max().item()
            
        print(f"L-BFGS Epoch [{epoch:04d}/{EPOCHS_LBFGS}] | Loss: {loss_val:.4e} | V_min: {v_min:.1f} | V_max: {v_max:.1f}")

elapsed = time.time() - start_time
print(f"\n[MLOps HPC] Treinamento Multi-Estágio concluído em {elapsed/60:.2f} minutos.")

In [ ]:
# ==============================================================================
# CELULA 20 [PIVOT HÍBRIDO]: FWI DETERMINÍSTICO (DEEPWAVE + AUTOGRAD)
# ==============================================================================
# DIAGRAMA DE FLUXO HPC: O PADRÃO OURO DA INDÚSTRIA (FDTD + ML)
# ------------------------------------------------------------------------------
#  [ Matriz de Velocidade (Geologia) ] ---> (Transposição Z,X -> X,Z)
#                 |
#                 v
#  +------------------------------------------------------------------------+
#  | MOTOR FDTD (DEEPWAVE) - LIVRE DE VIÉS ESPECTRAL                        |
#  | 1. Injeta Wavelet de Ricker nas posições das 5 Fontes.                 |
#  | 2. Propaga a onda no tempo usando Diferenças Finitas (Física Exata).   |
#  | 3. Extrai o Sismograma Sintético nos 70 Receptores.                    |
#  +------------------------------------------------------------------------+
#                 |
#                 v
#  [ Data Loss (MSE) ] ---> Compara com o Sismograma Real (OpenFWI)
#                 |
#                 v
#  [ Autograd (Backward) ] ---> Retropropaga o erro pelo motor FDTD
#                 |
#                 v
#  [ Otimizador (Adam) ] ---> Atualiza a Matriz de Velocidade
# ==============================================================================

import time
import torch
import torch.nn as nn
import deepwave

print("[MLOps HPC] Pivotando para Arquitetura Híbrida (Deepwave FDTD + PyTorch Autograd)...")

# ------------------------------------------------------------------------------
# 1. Compatibilidade Arquitetural (Mantendo o Dashboard Célula 18 funcional)
# ------------------------------------------------------------------------------
class DeepwaveVelocityModel(nn.Module):
    """
    O QUE FAZ: Armazena a geologia no formato [1, 1, NZ, NX] (padrão de imagens ML),
    mas fornece um método para entregar a matriz no formato (NX, NZ) exigido pelo Deepwave.
    
    PARA QUE SERVE: Garante que possamos usar o motor físico determinístico sem 
    quebrar o pipeline de Quality Assurance (QA) construído na Célula 18.
    """
    def __init__(self, nx: int, nz: int, initial_vel: float):
        super().__init__()
        self.grid = nn.Parameter(torch.ones(1, 1, nz, nx) * initial_vel)
        
    def get_deepwave_model(self) -> torch.Tensor:
        # Remove dimensões vazias e transpõe de (Z, X) para (X, Z)
        return self.grid.squeeze().T

# Instancia o Produto Comercial
vel_model = DeepwaveVelocityModel(nx=NX, nz=NZ, initial_vel=1500.0).to(device)

# ------------------------------------------------------------------------------
# 2. Engenharia de Dados para o Deepwave (Topologia OpenFWI)
# ------------------------------------------------------------------------------
# O OpenFWI fornece os dados como [Tiros, Tempo, Receptores].
# O Deepwave exige [Tiros, Receptores, Tempo]. Precisamos transpor (eixos 1 e 2).
d_obs_true = torch.tensor(seismic_obs, dtype=torch.float32, device=device).transpose(1, 2)

# Mapeamento das 5 Fontes (Índices do Grid)
# X: Espaçadas uniformemente de 0 a 69. Z: Profundidade 1 (10 metros)
src_locs = torch.zeros(NUM_SHOTS, 1, 2, dtype=torch.long, device=device)
src_locs[:, 0, 0] = torch.linspace(0, NX - 1, NUM_SHOTS).long()
src_locs[:, 0, 1] = 1 

# Mapeamento dos 70 Receptores (Índices do Grid)
# X: Todos os pontos de 0 a 69. Z: Profundidade 1 (10 metros)
rec_locs = torch.zeros(NUM_SHOTS, NUM_REC, 2, dtype=torch.long, device=device)
rec_locs[:, :, 0] = torch.arange(NUM_REC).repeat(NUM_SHOTS, 1)
rec_locs[:, :, 1] = 1

# Geração da Wavelet de Ricker (Frequência central de 15Hz, padrão OpenFWI)
peak_freq = 15.0
src_amps = deepwave.wavelets.ricker(peak_freq, NT, DT, 1.0/peak_freq).repeat(NUM_SHOTS, 1, 1).to(device)

# ------------------------------------------------------------------------------
# 3. Motor de Treinamento FWI (Adam)
# ------------------------------------------------------------------------------
EPOCHS_FWI = 150
LR_FWI = 25.0  # Taxa agressiva, pois o gradiente FDTD é muito bem comportado

optimizer_fwi = torch.optim.Adam(vel_model.parameters(), lr=LR_FWI)

print(f"\n>>> INICIANDO INVERSÃO FWI DETERMINÍSTICA ({EPOCHS_FWI} Épocas) <<<")
start_time = time.time()

for epoch in range(1, EPOCHS_FWI + 1):
    optimizer_fwi.zero_grad()
    
    # Extrai o modelo no formato correto para o Deepwave
    v_dw = vel_model.get_deepwave_model()
    
    # Forward Pass (Física Exata)
    out = deepwave.scalar(
        v_dw, DX, DT, max_vel=4500.0,
        source_amplitudes=src_amps,
        source_locations=src_locs,
        receiver_locations=rec_locs,
        accuracy=8,
        pml_freq=peak_freq,
        pml_width=[20, 20, 20, 20]
    )
    
    # Extrai o sismograma sintético
    d_syn = out[-1]
    
    # Calcula o resíduo (Data Loss)
    loss = torch.nn.MSELoss()(d_syn, d_obs_true)
    
    # Backward Pass (Estado Adjunto via Autograd)
    loss.backward()
    
    # Atualiza a geologia
    optimizer_fwi.step()
    
    # Trava de Segurança Geológica
    with torch.no_grad():
        vel_model.grid.clamp_(min=1400.0, max=4500.0)
        v_min = vel_model.grid.min().item()
        v_max = vel_model.grid.max().item()
        
    if epoch % 10 == 0 or epoch == 1:
        print(f"FWI Epoch [{epoch:03d}/{EPOCHS_FWI}] | Loss: {loss.item():.4e} | V_min: {v_min:.1f} | V_max: {v_max:.1f}")

elapsed = time.time() - start_time
print(f"\n[MLOps HPC] Inversão FWI concluída em {elapsed/60:.2f} minutos.")

In [ ]:
# ==============================================================================
# CELULA 19: DASHBOARD DE INFERÊNCIA E QUALITY ASSURANCE (QA)
# ==============================================================================
# DIAGRAMA DE INFERÊNCIA E QA (QUALITY ASSURANCE)
# ------------------------------------------------------------------------------
#  [ GPU VRAM ]
#       |
#       +---> vel_model.grid.detach().cpu() ---> [ Matriz Invertida (2D) ]
#                                                       |
#  [ Memória RAM ]                                      v
#       |                                     [ Comparação Forense ]
#       +---> true_velocity (Ground Truth) -------->    |
#                                                       v
#                                             +--------------------+
#                                             | 1. Modelo Real     |
#                                             | 2. Modelo PINN     |
#                                             | 3. Mapa de Erro    |
#                                             | 4. Perfil 1D (Poço)|
#                                             +--------------------+
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt

def generate_qa_dashboard(vel_model: torch.nn.Module, true_vel: np.ndarray):
    """
    O QUE FAZ:
    Extrai o grid de velocidades treinado da GPU, converte para NumPy e plota 
    um painel comparativo com 4 visões: Ground Truth, Predição, Erro Absoluto 
    e um Perfil de Poço 1D (Trace).

    PARA QUE SERVE:
    Auditoria visual do produto final. Na indústria, não entregamos "Losses" 
    para o cliente, entregamos imagens da subsuperfície. O Perfil 1D simula 
    a perfuração de um poço exploratório no centro do modelo para verificar 
    se a PINN acertou a profundidade exata das camadas geológicas.
    """
    print("[Inferência] Extraindo o Produto Comercial da GPU...")
    
    # 1. Extração Segura (Descolamento do Grafo Computacional)
    # .detach() corta a ligação com o Autograd.
    # .cpu() move da VRAM para a RAM.
    # .squeeze() remove as dimensões de Batch e Canal [1, 1, NZ, NX] -> [NZ, NX]
    inverted_vel = vel_model.grid.detach().cpu().squeeze().numpy()
    
    # 2. Cálculo do Erro Absoluto
    error_map = np.abs(true_vel - inverted_vel)
    
    # 3. Extração do Perfil 1D (Simulação de Poço no centro do eixo X)
    center_x = NX // 2
    trace_true = true_vel[:, center_x]
    trace_inv = inverted_vel[:, center_x]
    depth_axis = np.arange(NZ) * DX
    
    # 4. Renderização do Dashboard
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("DELTA-PINN ENTERPRISE: Relatório de Quality Assurance (QA)", fontsize=16, fontweight='bold')
    
    # Escala de cores unificada para os modelos de velocidade
    vmin = min(true_vel.min(), inverted_vel.min())
    vmax = max(true_vel.max(), inverted_vel.max())
    
    # --- Plot 1: Ground Truth ---
    im0 = axes[0, 0].imshow(true_vel, cmap='jet', vmin=vmin, vmax=vmax, aspect='auto')
    axes[0, 0].set_title("Modelo Verdadeiro (Ground Truth)")
    axes[0, 0].set_ylabel("Profundidade (Z)")
    fig.colorbar(im0, ax=axes[0, 0], label="Velocidade (m/s)")
    
    # --- Plot 2: Modelo Invertido (PINN) ---
    im1 = axes[0, 1].imshow(inverted_vel, cmap='jet', vmin=vmin, vmax=vmax, aspect='auto')
    axes[0, 1].set_title(f"Modelo Invertido (PINN - {EPOCHS} Épocas)")
    fig.colorbar(im1, ax=axes[0, 1], label="Velocidade (m/s)")
    
    # --- Plot 3: Mapa de Erro Absoluto ---
    im2 = axes[1, 0].imshow(error_map, cmap='magma', aspect='auto')
    axes[1, 0].set_title("Mapa de Erro Absoluto |True - PINN|")
    axes[1, 0].set_xlabel("Distância (X)")
    axes[1, 0].set_ylabel("Profundidade (Z)")
    fig.colorbar(im2, ax=axes[1, 0], label="Erro (m/s)")
    
    # --- Plot 4: Perfil de Poço 1D ---
    axes[1, 1].plot(trace_true, depth_axis, 'k-', linewidth=2, label="Perfil Real")
    axes[1, 1].plot(trace_inv, depth_axis, 'r--', linewidth=2, label="Perfil PINN")
    axes[1, 1].invert_yaxis() # Profundidade cresce para baixo
    axes[1, 1].set_title(f"Perfil de Poço 1D (X = {center_x * DX} m)")
    axes[1, 1].set_xlabel("Velocidade (m/s)")
    axes[1, 1].set_ylabel("Profundidade (m)")
    axes[1, 1].legend()
    axes[1, 1].grid(True, linestyle=':', alpha=0.7)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.show()

# Executa a geração do Dashboard
generate_qa_dashboard(vel_model, true_velocity)

In [ ]:
# ==============================================================================
# CELULA 21: ANÁLISE FÍSICA DO OPENFWI (FORWARD E GRADIENTE INICIAL)
# ==============================================================================
# DIAGRAMA DE INSPEÇÃO FÍSICA: O NASCIMENTO DO GRADIENTE
# ------------------------------------------------------------------------------
#  [ Modelo OpenFWI (70x70) ]       [ Modelo Cego (1500 m/s) ]
#             |                                 |
#             v                                 v
#  [ FDTD Forward (Deepwave) ]       [ FDTD Forward (Deepwave) ]
#             |                                 |
#             v                                 v
#  [ Sismograma Real (d_obs) ]       [ Sismograma Sintético (d_syn) ]
#             \                                 /
#              \                               /
#               +-----> [ Resíduo (MSE) ] <---+
#                               |
#                               v
#                 [ Autograd (loss.backward) ]
#                               |
#                               v
#             [ Mapa do Gradiente Inicial (Z, X) ]
#  (Revela a iluminação da onda e a origem dos Source Footprints)
# ==============================================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
import deepwave

print("[Física Forense] Configurando o experimento OpenFWI para extração de Gradiente...")

# ------------------------------------------------------------------------------
# 1. Preparação Topológica (NumPy -> PyTorch Deepwave)
# Deepwave exige o formato (NX, NZ). O OpenFWI original é (NZ, NX).
# Como é 70x70, a transposição (.T) alinha os eixos físicos corretamente.
# ------------------------------------------------------------------------------
v_true = torch.tensor(true_velocity, dtype=torch.float32, device=device).T
v_homo = (torch.ones(NX, NZ, dtype=torch.float32, device=device) * 1500.0).requires_grad_(True)

# ------------------------------------------------------------------------------
# 2. Geometria de Aquisição (5 Tiros, 70 Receptores)
# ------------------------------------------------------------------------------
# Fontes: 5 posições espaçadas no eixo X, profundidade Z = 1 (10 metros)
src_locs = torch.zeros(NUM_SHOTS, 1, 2, dtype=torch.long, device=device)
src_locs[:, 0, 0] = torch.linspace(0, NX - 1, NUM_SHOTS).long()
src_locs[:, 0, 1] = 1 

# Receptores: 70 posições cobrindo todo o eixo X, profundidade Z = 1
rec_locs = torch.zeros(NUM_SHOTS, NUM_REC, 2, dtype=torch.long, device=device)
rec_locs[:, :, 0] = torch.arange(NUM_REC).repeat(NUM_SHOTS, 1)
rec_locs[:, :, 1] = 1

# Wavelet de Ricker (15 Hz)
peak_freq = 15.0
src_amps = deepwave.wavelets.ricker(peak_freq, NT, DT, 1.0/peak_freq).repeat(NUM_SHOTS, 1, 1).to(device)

# ------------------------------------------------------------------------------
# 3. Modelagem Direta (Forward Pass)
# ------------------------------------------------------------------------------
print("[Física Forense] Propagando ondas no Modelo Verdadeiro e no Modelo Cego...")

# Sismograma Real
out_true = deepwave.scalar(
    v_true, DX, DT, max_vel=4500.0,
    source_amplitudes=src_amps, source_locations=src_locs, receiver_locations=rec_locs,
    accuracy=8, pml_freq=peak_freq, pml_width=[20, 20, 20, 20]
)
d_obs = out_true[-1].detach()

# Sismograma Sintético (Background)
out_homo = deepwave.scalar(
    v_homo, DX, DT, max_vel=4500.0,
    source_amplitudes=src_amps, source_locations=src_locs, receiver_locations=rec_locs,
    accuracy=8, pml_freq=peak_freq, pml_width=[20, 20, 20, 20]
)
d_syn = out_homo[-1]

# ------------------------------------------------------------------------------
# 4. Extração do Gradiente (Backward Pass)
# ------------------------------------------------------------------------------
print("[Física Forense] Calculando o Gradiente FWI via Estado Adjunto (Autograd)...")
loss = torch.nn.MSELoss()(d_syn, d_obs)
loss.backward()

# Extrai o gradiente, move para CPU e transpõe de volta para (Z, X) para plotagem
grad_map = v_homo.grad.detach().cpu().T.numpy()

# Condicionamento visual (Clip) para não ofuscar o gradiente com a singularidade da fonte
clip_val = np.percentile(np.abs(grad_map), 99.5)

# ------------------------------------------------------------------------------
# 5. Dashboard de Física Forense
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("DELTA-PINN ENTERPRISE: Inspeção do Gradiente Inicial (OpenFWI)", fontsize=14, fontweight='bold')

# Plot 1: Sismograma Real (Tiro Central)
shot_idx = NUM_SHOTS // 2
trace_obs = d_obs[shot_idx].cpu().numpy().T
vmax_trace = np.max(np.abs(trace_obs))
im0 = axes[0].imshow(trace_obs, aspect='auto', cmap='gray', vmin=-vmax_trace, vmax=vmax_trace)
axes[0].set_title(f"Sismograma Real (Tiro {shot_idx + 1})")
axes[0].set_xlabel("Receptores (X)")
axes[0].set_ylabel("Tempo (Amostras)")

# Plot 2: Resíduo (Tiro Central)
trace_res = (d_syn[shot_idx].detach() - d_obs[shot_idx]).cpu().numpy().T
im1 = axes[1].imshow(trace_res, aspect='auto', cmap='gray', vmin=-vmax_trace, vmax=vmax_trace)
axes[1].set_title(f"Resíduo FWI (Tiro {shot_idx + 1})")
axes[1].set_xlabel("Receptores (X)")

# Plot 3: Mapa do Gradiente FWI
im2 = axes[2].imshow(grad_map, aspect='auto', cmap='seismic', vmin=-clip_val, vmax=clip_val)
axes[2].set_title("Gradiente FWI Inicial (Z, X)")
axes[2].set_xlabel("Distância (X)")
axes[2].set_ylabel("Profundidade (Z)")
fig.colorbar(im2, ax=axes[2], label="Magnitude do Gradiente")

# Marca as fontes no mapa de gradiente para evidenciar o "Source Footprint"
axes[2].plot(src_locs[:, 0, 0].cpu().numpy(), src_locs[:, 0, 1].cpu().numpy(), 'y*', markersize=12, label="Fontes")
axes[2].legend(loc="lower right")

plt.tight_layout()
plt.show()